# Classification — Solutions

<a target="_blank" href="https://colab.research.google.com/github/LuWidme/uk259/blob/rework/solutions/06_Classification_solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ~90 min · Day 3

**Important:** Try the exercises in `demos/06_Classification.ipynb` yourself first.

Reference solutions with short explanations — your own version may differ and still be correct.

In [ ]:
# === COURSE SETUP — run this cell first! ===
%pip install -q numpy pandas matplotlib seaborn scikit-learn

import os, urllib.request
DATA_URL = "https://raw.githubusercontent.com/LuWidme/uk259/rework/datasets/"
for folder in ("datasets", os.path.join("..", "datasets")):
    os.makedirs(folder, exist_ok=True)
    for fname in ['titanic.csv', 'melb_data.csv', 'Company_data.csv']:
        path = os.path.join(folder, fname)
        if not os.path.exists(path):
            urllib.request.urlretrieve(DATA_URL + fname, path)
print("Setup complete — you are ready to go!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Iris split (used by Exercise 2)
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris)
titanic = pd.read_csv('../datasets/titanic.csv')
print('Ready.')

### Task 1: Implement KNN from Scratch

In [ ]:
def knn_predict(new_point, experience_data, experience_labels, k):
    # 1) distance from new_point to every training point
    distances = np.sqrt(np.sum((experience_data - new_point) ** 2, axis=1))
    # 2) indices of the k closest points
    nearest = np.argsort(distances)[:k]
    # 3) majority vote among their labels
    nearest_labels = experience_labels[nearest]
    values, counts = np.unique(nearest_labels, return_counts=True)
    return values[np.argmax(counts)]

# quick check on iris
pred = knn_predict(X_test_iris[0], X_train_iris, y_train_iris, k=5)
print('Predicted:', pred, '| Actual:', y_test_iris[0])

## Part 5: Exercises

### Exercise 1: Titanic Survival Prediction

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'Siblings/Spouses Aboard',
            'Parents/Children Aboard', 'Fare']
df = titanic[features + ['Survived']].copy()
df['Age'] = df['Age'].fillna(df['Age'].median())
df = pd.get_dummies(df, columns=['Sex'], drop_first=True)

X = df.drop('Survived', axis=1)
y = df['Survived']
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('Train/test:', X_train_t.shape, X_test_t.shape)

In [ ]:
# Scale (important for KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_t)
X_test_scaled = scaler.transform(X_test_t)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train_t)
knn_acc = accuracy_score(y_test_t, knn.predict(X_test_scaled))

dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train_t, y_train_t)   # trees don't need scaling
dt_acc = accuracy_score(y_test_t, dt.predict(X_test_t))

print(f'KNN accuracy:           {knn_acc:.3f}')
print(f'Decision Tree accuracy: {dt_acc:.3f}')
ConfusionMatrixDisplay.from_predictions(y_test_t, dt.predict(X_test_t))
plt.title('Decision Tree — Titanic'); plt.show()

*Decision Trees and KNN usually land around 78–82% here. Trees often edge ahead and are easier to explain.*

### Exercise 2: KNN Parameter Tuning

In [ ]:
k_values = range(1, 16)
train_accuracies, test_accuracies = [], []
for k in k_values:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_iris, y_train_iris)
    train_accuracies.append(m.score(X_train_iris, y_train_iris))
    test_accuracies.append(m.score(X_test_iris, y_test_iris))

best_k = k_values[int(np.argmax(test_accuracies))]
plt.plot(k_values, train_accuracies, 'o-', label='train')
plt.plot(k_values, test_accuracies, 's-', label='test')
plt.xlabel('k'); plt.ylabel('accuracy'); plt.legend(); plt.title('KNN: choosing k'); plt.show()
print('Best k on test set:', best_k)